# Field Scan : modèle maladies v2 (Kaggle)

Classification de photos de feuilles de bananier en 3 classes (saine, Black Sigatoka, Fusarium), avec une réponse « inconnu » quand la photo ne ressemble pas à ce que le modèle connaît.

| Rôle | Jeu | Pourquoi |
| --- | --- | --- |
| Entraînement | Tanzanie 2023 + Pérou (saines et Black Sigatoka) | 2 pays : le modèle ne peut pas se reposer sur un seul téléphone ou une seule lumière |
| Validation | Tanzanie 2025 | choix de la meilleure époque et des deux seuils |
| Test | Bangladesh | utilisé **une seule fois**, à la fin ; cordana et pestalotiopsis, jamais vues, doivent donner « inconnu » |

Ce qui change par rapport à la v1 :
1. le Pérou rejoint l'entraînement ;
2. des augmentations plus fortes (cadrage, rotation, couleurs, flou) ;
3. un **seuil d'alerte maladie** réglé pour retrouver au moins 95 % des feuilles malades de la validation (une feuille malade ratée est l'erreur la plus grave) ;
4. le rejet « inconnu » utilise la confiance du modèle **et** la distance entre la photo et les photos d'entraînement.

Le réseau reste le même que la v1 (MobileNetV3-Large), pour que les écarts viennent de ces changements.

**Avant de lancer :** Settings → Accelerator → **GPU T4**, Internet **On**, et le jeu `feuilles` ajouté dans Input (zip de `data/traite/feuilles` : `images/`, `images.csv`, `doublons_tanzanie.txt`, `doublons_bangladesh.txt`).

**Plusieurs essais :** changer `NOM_ESSAI` et les réglages de la cellule 2, relancer, comparer les résultats de **validation**. Laisser `LANCER_TEST = False` tant que le modèle final n'est pas choisi.

In [ ]:
# 1. GPU et emplacement des données
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path

trouves = sorted(Path('/kaggle/input').rglob('images.csv'))
assert trouves, "images.csv introuvable : ajouter le jeu 'feuilles' dans Input"
RACINE = trouves[0].parent
print('Données :', RACINE)
print(sorted(p.name for p in RACINE.iterdir()))

In [ ]:
# 2. Réglages (à changer d'un essai à l'autre)
import random, json, time
import numpy as np, pandas as pd, torch

NOM_ESSAI = 'v2_essai1'            # nom du dossier de résultats
CLASSES = ['saine', 'black_sigatoka', 'fusarium']
EPOQUES = 10
TAILLE_LOT = 64
TAUX_APPRENTISSAGE = 3e-4
POIDS_PEROU = 3                     # une photo du Pérou est tirée 3 fois plus souvent (le jeu est petit)
OBJECTIF_RAPPEL_MALADE = 0.95       # seuil d'alerte : retrouver au moins 95 % des feuilles malades en validation
ACCEPTATION_VALIDATION = 0.95       # seuil « inconnu » : garder 95 % des photos de validation
LANCER_TEST = False                 # True seulement pour le test final sur le Bangladesh, une seule fois
GRAINE = 0

random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
appareil = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Calcul sur :', appareil, torch.cuda.get_device_name(0) if appareil == 'cuda' else '')
SORTIE = Path('/kaggle/working') / NOM_ESSAI
SORTIE.mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Découpage : par pays et par campagne, jamais au hasard
images = pd.read_csv(RACINE / 'images.csv')

def lire_liste(nom):
    # Entrées comparées sans extension : 'HEALTHY_1/Image_1094'
    return {l.strip().rsplit('.', 1)[0] for l in open(RACINE / nom, encoding='utf-8')
            if l.strip() and not l.startswith('#')}

def source_courte(chemin):
    # 'tanzanie_2023_dataverse/HEALTHY_1/HEALTHY_1/Image_1094.jpg' -> 'HEALTHY_1/Image_1094'
    return '/'.join(chemin.split('/')[-2:]).rsplit('.', 1)[0]

courte = images.source.map(source_courte)
tanz = images.jeu.str.startswith('tanzanie')
retire = (tanz & courte.isin(lire_liste('doublons_tanzanie.txt'))) | \
         ((images.jeu == 'bangladesh') & courte.isin(lire_liste('doublons_bangladesh.txt')))
print('Copies écartées :', images[retire].jeu.value_counts().to_dict())
images = images[~retire]

entrainement = images[images.jeu.isin(['tanzanie_2023', 'perou'])]
validation = images[images.jeu == 'tanzanie_2025']
test = images[images.jeu == 'bangladesh']

for nom, df in [('entraînement', entrainement), ('validation', validation), ('test', test)]:
    print(f'{nom:13s} {len(df):6d} images  ', df.groupby(['jeu', 'classe']).size().to_dict())
assert len(test) == 893, 'le test doit compter 893 images après retrait des doublons'
assert set(entrainement.classe) <= set(CLASSES) and set(validation.classe) <= set(CLASSES)

In [ ]:
# 4. Chargement des images ; augmentations fortes pour l'entraînement seulement
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T

NORMALISATION = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # celle d'ImageNet
aug_entrainement = T.Compose([
    T.RandomResizedCrop(224, scale=(0.4, 1.0), ratio=(0.8, 1.25)),   # zoom et cadrage variés
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.04),  # autre lumière, autre téléphone
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),        # photo un peu floue
    T.ToTensor(), NORMALISATION,
])
aug_evaluation = T.Compose([T.ToTensor(), NORMALISATION])

class Feuilles(Dataset):
    def __init__(self, df, transformation):
        self.chemins = list(df.image)
        self.etiquettes = [CLASSES.index(c) if c in CLASSES else -1 for c in df.classe]
        self.transformation = transformation
    def __len__(self):
        return len(self.chemins)
    def __getitem__(self, i):
        image = Image.open(RACINE / self.chemins[i]).convert('RGB')
        return self.transformation(image), self.etiquettes[i]

poids = np.where(entrainement.jeu == 'perou', POIDS_PEROU, 1.0)
tirage = WeightedRandomSampler(torch.tensor(poids, dtype=torch.double), num_samples=len(entrainement), replacement=True)
charge_entrainement = DataLoader(Feuilles(entrainement, aug_entrainement), TAILLE_LOT, sampler=tirage, num_workers=4)
charge_entrainement_sans_aug = DataLoader(Feuilles(entrainement, aug_evaluation), TAILLE_LOT, num_workers=4)
charge_validation = DataLoader(Feuilles(validation, aug_evaluation), TAILLE_LOT, num_workers=4)
charge_test = DataLoader(Feuilles(test, aug_evaluation), TAILLE_LOT, num_workers=4)
print(f"Part du Pérou dans un tour d'entraînement : {poids[entrainement.jeu.values == 'perou'].sum() / poids.sum():.0%}")

In [ ]:
# 5. Modèle : MobileNetV3-Large pré-entraîné sur ImageNet, dernière couche remplacée (3 classes)
#    Même réseau que la v1 ; léger, utilisable sans GPU.
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

modele = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
modele.classifier[3] = torch.nn.Linear(modele.classifier[3].in_features, len(CLASSES))
modele = modele.to(appareil)
print(f'{sum(p.numel() for p in modele.parameters()) / 1e6:.1f} millions de paramètres')

In [ ]:
# 6. Entraînement ; on garde l'époque qui a la meilleure F1 moyenne en validation
from sklearn.metrics import f1_score

def predire(charge):
    # Renvoie les probabilités, les vraies classes et le « résumé » de chaque photo
    # (sortie de l'avant-dernière couche, 1280 nombres) qui sert au rejet « inconnu ».
    modele.eval(); probas, vraies, resumes = [], [], []
    with torch.no_grad():
        for x, y in charge:
            x = x.to(appareil)
            f = modele.classifier[:3](torch.flatten(modele.avgpool(modele.features(x)), 1))
            probas.append(torch.softmax(modele.classifier[3](f), 1).cpu())
            resumes.append(f.cpu()); vraies.append(y)
    return torch.cat(probas).numpy(), torch.cat(vraies).numpy(), torch.cat(resumes).numpy()

optimiseur = torch.optim.AdamW(modele.parameters(), lr=TAUX_APPRENTISSAGE, weight_decay=1e-4)
planning = torch.optim.lr_scheduler.CosineAnnealingLR(optimiseur, T_max=EPOQUES)
perte = torch.nn.CrossEntropyLoss()
meilleure_f1, historique = -1, []
POIDS_MODELE = SORTIE / 'maladies_v2.pt'

for epoque in range(1, EPOQUES + 1):
    modele.train(); debut = time.time(); total = 0
    for x, y in charge_entrainement:
        x, y = x.to(appareil), y.to(appareil)
        optimiseur.zero_grad(); l = perte(modele(x), y); l.backward(); optimiseur.step()
        total += l.item() * len(y)
    planning.step()
    probas, vraies, _ = predire(charge_validation)
    f1 = f1_score(vraies, probas.argmax(1), average='macro')
    historique.append({'epoque': epoque, 'perte': total / len(charge_entrainement.dataset), 'f1_validation': f1})
    print(f'époque {epoque}  perte {historique[-1]["perte"]:.3f}  F1 validation {f1:.3f}  ({time.time() - debut:.0f} s)')
    assert np.isfinite(historique[-1]['perte']), 'perte NaN : entraînement à relancer'
    if f1 > meilleure_f1:
        meilleure_f1 = f1
        torch.save(modele.state_dict(), POIDS_MODELE)

modele.load_state_dict(torch.load(POIDS_MODELE))
print('Meilleure F1 en validation :', round(meilleure_f1, 3))

In [ ]:
# 7. Seuil d'alerte maladie (réglé sur la validation seulement)
#    p_malade = probabilité Black Sigatoka + probabilité Fusarium.
#    Si p_malade >= seuil, réponse = la maladie la plus probable ; sinon « saine ».
#    On prend le seuil le plus haut qui retrouve encore au moins 95 % des feuilles malades.
probas_val, vraies_val, resumes_val = predire(charge_validation)
malade_val = vraies_val > 0
p_malade_val = probas_val[:, 1:].sum(1)

candidats = np.sort(np.unique(np.round(p_malade_val, 4)))[::-1]
SEUIL_ALERTE = 0.5
for s in candidats:
    if (p_malade_val[malade_val] >= s).mean() >= OBJECTIF_RAPPEL_MALADE:
        SEUIL_ALERTE = float(s); break
SEUIL_ALERTE = min(SEUIL_ALERTE, 0.5)   # jamais plus strict que le choix « classe la plus probable »

def decider(probas):
    p_malade = probas[:, 1:].sum(1)
    maladie = 1 + probas[:, 1:].argmax(1)
    return np.where(p_malade >= SEUIL_ALERTE, maladie, 0)

def bilan(vraies, choix, titre):
    malade = vraies > 0
    print(f'--- {titre}')
    print(f'  rappel malades (feuilles malades retrouvées)  : {(choix[malade] > 0).mean():.3f}')
    print(f'  fausses alertes (saines dites malades)         : {(choix[~malade] > 0).mean():.3f}')
    print(f'  F1 moyenne des 3 classes                       : {f1_score(vraies, choix, average="macro"):.3f}')

print(f'Seuil d\'alerte : {SEUIL_ALERTE:.3f}')
bilan(vraies_val, probas_val.argmax(1), 'validation, sans seuil (comme la v1)')
bilan(vraies_val, decider(probas_val), 'validation, avec seuil d\'alerte')
print(pd.crosstab(pd.Series(np.array(CLASSES)[vraies_val], name='réel'),
                  pd.Series(np.array(CLASSES)[decider(probas_val)], name='réponse')))

In [ ]:
# 8. Rejet « inconnu » (réglé sur la validation seulement, aucune photo du test)
#    Deux mesures :
#    - confiance : la probabilité la plus haute donnée par le modèle ;
#    - distance : à quel point le résumé de la photo est loin du résumé moyen de chaque classe
#      d'entraînement (distance de Mahalanobis, on garde la classe la plus proche).
#    La photo est « inconnue » si la confiance est trop basse OU la distance trop grande.
#    Les deux seuils sont réglés ensemble pour garder 95 % des photos de validation.
from sklearn.covariance import LedoitWolf

_, vraies_ent, resumes_ent = predire(charge_entrainement_sans_aug)
centres = np.stack([resumes_ent[vraies_ent == k].mean(0) for k in range(len(CLASSES))])
ecarts = resumes_ent - centres[vraies_ent]
precision_cov = LedoitWolf().fit(ecarts).precision_.astype(np.float32)

def distance(resumes):
    d = []
    for c in centres:
        e = resumes - c
        d.append(np.einsum('ij,jk,ik->i', e, precision_cov, e))
    return np.min(d, 0)

conf_val, dist_val = probas_val.max(1), distance(resumes_val)

# On cherche la même part q pour les deux mesures, telle que l'union rejette 5 % de la validation
bas, haut = 0.0, 1 - ACCEPTATION_VALIDATION
for _ in range(40):
    q = (bas + haut) / 2
    rejet = (conf_val < np.quantile(conf_val, q)) | (dist_val > np.quantile(dist_val, 1 - q))
    bas, haut = (q, haut) if rejet.mean() < 1 - ACCEPTATION_VALIDATION else (bas, q)
SEUIL_CONFIANCE = float(np.quantile(conf_val, q))
SEUIL_DISTANCE = float(np.quantile(dist_val, 1 - q))

def est_inconnu(probas, resumes):
    return (probas.max(1) < SEUIL_CONFIANCE) | (distance(resumes) > SEUIL_DISTANCE)

inc_val = est_inconnu(probas_val, resumes_val)
print(f'Seuil de confiance : {SEUIL_CONFIANCE:.3f}   seuil de distance : {SEUIL_DISTANCE:.1f}')
print(f'Photos de validation dites « inconnues » : {inc_val.mean():.1%} '
      f'(confiance seule {(conf_val < SEUIL_CONFIANCE).mean():.1%}, distance seule {(dist_val > SEUIL_DISTANCE).mean():.1%})')
# Une feuille malade dite « inconnue » part en vérification humaine : elle n'est pas ratée.
choix_val = decider(probas_val)
print(f'Malades ni signalées ni rejetées (vraiment ratées) : {((choix_val == 0) & ~inc_val & malade_val).sum()} '
      f'sur {malade_val.sum()}')

In [ ]:
# 9. Sauvegarde de l'essai (poids, seuils, statistiques pour le rejet, résultats de validation)
np.savez(SORTIE / 'rejet_inconnu.npz', centres=centres, precision_cov=precision_cov)
choix_val = decider(probas_val)
reglages = {
    'nom_essai': NOM_ESSAI, 'classes': CLASSES, 'epoques': EPOQUES, 'taux_apprentissage': TAUX_APPRENTISSAGE,
    'poids_perou': POIDS_PEROU, 'historique': historique, 'meilleure_f1_validation': meilleure_f1,
    'seuil_alerte': SEUIL_ALERTE, 'seuil_confiance': SEUIL_CONFIANCE, 'seuil_distance': SEUIL_DISTANCE,
    'validation': {
        'rappel_malades': float((choix_val[malade_val] > 0).mean()),
        'fausses_alertes_saines': float((choix_val[~malade_val] > 0).mean()),
        'f1_avec_seuil': float(f1_score(vraies_val, choix_val, average='macro')),
        'part_inconnu': float(inc_val.mean()),
    },
}
json.dump(reglages, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(reglages['validation'], indent=2))
print('Écrit dans', SORTIE, ':', sorted(p.name for p in SORTIE.iterdir()))

## Test final sur le Bangladesh

À lancer **une seule fois**, quand l'essai final est choisi d'après la validation : mettre `LANCER_TEST = True` dans la cellule 2, relancer tout le notebook, puis ne plus rien changer.

In [ ]:
# 10. Test sur le Bangladesh : une seule fois, après tous les réglages
assert LANCER_TEST, 'Test bloqué : mettre LANCER_TEST = True seulement pour le test final'
from sklearn.metrics import roc_auc_score

probas_test, _, resumes_test = predire(charge_test)
reponse = np.array(CLASSES)[decider(probas_test)].astype(object)
reponse[est_inconnu(probas_test, resumes_test)] = 'inconnu'
t = test.assign(reponse=reponse, confiance=probas_test.max(1), distance=distance(resumes_test))

# La sigatoka du Bangladesh compte comme Black Sigatoka (forme non précisée par le jeu)
print('Matrice de confusion (lignes : classe réelle, colonnes : réponse du modèle)')
print(pd.crosstab(t.classe, t.reponse))

sig, saines = t[t.classe == 'sigatoka'], t[t.classe == 'saine']
connues = t[t.classe.isin(['saine', 'sigatoka'])]
inconnues = t[t.classe.isin(['cordana', 'pestalotiopsis'])]
attendu = connues.classe.map({'saine': 'saine', 'sigatoka': 'black_sigatoka'})
est_inc = t.classe.isin(['cordana', 'pestalotiopsis'])
resultats = {
    'exactitude_classes_connues': float((connues.reponse == attendu).mean()),
    'rappel_sigatoka': float((sig.reponse == 'black_sigatoka').mean()),
    'sigatoka_ratees_dites_saines': float((sig.reponse == 'saine').mean()),
    'fausses_alertes_sur_saines': float(saines.reponse.isin(['black_sigatoka', 'fusarium']).mean()),
    'rejet_inconnu': float((inconnues.reponse == 'inconnu').mean()),
    'rejet_a_tort_classes_connues': float((connues.reponse == 'inconnu').mean()),
    # Capacité à séparer connu / inconnu, sans seuil (1 = parfait, 0,5 = hasard)
    'auroc_inconnu_confiance': float(roc_auc_score(est_inc, -t.confiance)),
    'auroc_inconnu_distance': float(roc_auc_score(est_inc, t.distance)),
}
for k, v in resultats.items():
    print(f'{k:32s} {v:.3f}')

reglages['test_bangladesh'] = resultats
json.dump(reglages, open(SORTIE / 'resultats.json', 'w'), indent=2, ensure_ascii=False)
t.to_csv(SORTIE / 'reponses_test.csv', index=False)
print('Écrit dans', SORTIE)